In [0]:
from pathlib import Path

In [0]:
dbutils.widgets.text("raw_root", "/Volumes/workspace/raw/proyecto", "Raw Root")
raw_root = Path(dbutils.widgets.get("raw_root"))

In [0]:
%sql
DROP TABLE IF EXISTS workspace.bronze.data_ubigeo_1;
DROP TABLE IF EXISTS workspace.bronze.data_ubigeo_2;
DROP TABLE IF EXISTS workspace.bronze.equivalencia_ubigeo; 

In [0]:
%sql
CREATE TABLE workspace.bronze.data_ubigeo_1 (
    ubigeo_Inei STRING,
    distrito STRING,
    provincia STRING,
    departamento STRING,
    capital_legal STRING,
    categoria STRING,
    altitud STRING,
    poblacion STRING,
    latitud STRING,
    longitud STRING
) USING DELTA;

In [0]:
%sql
CREATE TABLE workspace.bronze.data_ubigeo_2 (
    ubigeo_reniec STRING,
    distrito STRING,
    provincia STRING,
    departamento STRING,
    Superficie STRING,
    latitud STRING,
    longitud STRING
) USING DELTA;

In [0]:
%sql
CREATE TABLE workspace.bronze.equivalencia_ubigeo (
    ubigeo_Inei STRING,
    ubigeo_reniec STRING,
    distrito STRING,
    provincia STRING,
    departamento STRING
) USING DELTA;


In [0]:
# 1. data_ubigeo_1.csv
df_ubigeo1 = spark.read.csv(
    f"{raw_root}/data_ubigeo_1.csv",sep=";", header=True, inferSchema=False
)

df_ubigeo1 = (
    df_ubigeo1
    .withColumnRenamed("categoría", "categoria")
    .withColumnRenamed("población", "poblacion")
)

df_ubigeo1.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.data_ubigeo_1")

# 2. data_ubigeo_2.csv
df_ubigeo2 = spark.read.csv(
     f"{raw_root}/data_ubigeo_2.csv",sep=";", header=True, inferSchema=False, encoding="iso-8859-1"
)
df_ubigeo2.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.data_ubigeo_2")

# 3. equivalencia_ubigeo_inei_reniec.csv
df_equiv = spark.read.csv(
     f"{raw_root}/equivalencia_ubigeo_inei_reniec.csv",sep=";", header=True, inferSchema=False
)
df_equiv.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.equivalencia_ubigeo")